# 4. Optimise an index against real constraints

Needs the optimiser extra:

```
pip install "py-beacon[optimise]"
```

## What to look at

**Not the weights. The binding constraints.**

A constraint that bit changed the answer. One that did not was never relevant.
An optimiser that reports only weights leaves you unable to tell which was
which — and the whole reason to inspect a solution is to understand what shaped
it.

The last section makes that concrete by relaxing each constraint in turn and
watching what moves.

## Setup

In [ ]:
import logging

import pandas as pd

from beacon.index.calculation import IndexCalculator
from beacon.index.constructor import IndexDefinition
from beacon.index.methodology import MarketCapWeighted
from beacon.optimise import (
    FullInvestment,
    GroupBounds,
    PositionBounds,
    minimise_tracking_error,
)
from beacon.risk import estimate_risk_model, risk_contributions
from beacon.synthetic import SyntheticConfig, generate

logging.basicConfig(level=logging.ERROR,
                    format="%(levelname)s %(name)s: %(message)s")

pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## Step 1 — the target to track

A cap-weighted index with a 10% cap, exactly as in notebook 01. The optimiser's
job is to stay close to this while respecting limits the index itself does not
have.

In [ ]:
CONFIG = SyntheticConfig(assets=40,
                         start="2021-01-04",
                         end="2024-12-31",
                         seed=3)

dataset = generate(CONFIG)

definition = IndexDefinition(
    index_id="TARGET",
    index_name="TARGET Index",
    base_date=CONFIG.start,
    base_value=1000.0,
    currency=CONFIG.currency,
    eligibility_rules=[],
    weighting_scheme=MarketCapWeighted(use_free_float=True),
    rebalancing_frequency="QUARTERLY",
    universe_identifiers=list(dataset.universe.index),
    max_constituent_weight=0.10)

index = IndexCalculator(definition, dataset.fetcher()).run(
    start_date=CONFIG.start, end_date=CONFIG.end)

target = index.weight_snapshots[max(index.weight_snapshots)]

pd.Series(target).sort_values(ascending=False).head(8).to_frame(
    "target weight").style.format("{:.2%}")

## Step 2 — the risk model

Tracking error is a quadratic form in the covariance matrix, so the optimiser
cannot start without one.

**Shrinkage is on by default**, and this is the part worth understanding. A
sample covariance is noisiest in its smallest eigenvalues — and those are
exactly what an optimiser inverts. The raw estimate produces portfolios that
look brilliant in sample and fall apart out of it, because the optimiser
loads onto directions the data barely measured.

In [ ]:
risk = estimate_risk_model(dataset.returns)

pd.Series({"assets": f"{risk.diagnostics.assets}",
           "observations": f"{risk.diagnostics.observations}",
           "shrinkage": f"{risk.diagnostics.intensity:.3f} toward {risk.diagnostics.target}",
           "avg correlation": f"{risk.diagnostics.average_correlation:.3f}",
           "condition number": f"{risk.diagnostics.condition_number:.1f}",
           "positive semi-definite": f"{risk.diagnostics.positive_semi_definite!s}"},
          name="risk model")

The condition number is the diagnostic to watch. A high one means the matrix is
close to singular, and the weights the optimiser returns will be exquisitely
sensitive to inputs nobody estimated well.

## Step 3 — the constraints

Three of them. The last two rows show what the target already looks like
against each limit, so you can predict which will bite before solving:

- its largest name is at the index's own 10% cap, well **above** the 6%
  position limit — so that constraint must bind;
- its largest sector sits **below** the 25% group limit — so that one has
  nothing to do.

One of each is deliberate. A notebook where every constraint binds cannot show
you the difference between a constraint that shaped the answer and one that was
merely present.

In [ ]:
MAX_POSITION = 0.06
MAX_SECTOR = 0.25

sectors = dataset.universe.groupby("SECTOR").groups
largest_sector = max(sectors, key=lambda name: len(sectors[name]))
members = [str(name) for name in sectors[largest_sector]]

constraints = [
    FullInvestment(),
    PositionBounds(0.0, MAX_POSITION),
    GroupBounds(largest_sector, members, maximum=MAX_SECTOR),
]

sector_weight = sum(target.get(name, 0.0) for name in members)

pd.Series({"full investment": "weights sum to 1",
           "position bounds": f"0% to {MAX_POSITION:.0%} per name",
           "group bounds": f"{largest_sector} at most {MAX_SECTOR:.0%} ({len(members)} names)",
           "": "",
           "target's largest name": f"{max(target.values()):.2%}",
           f"target's {largest_sector}": f"{sector_weight:.2%}"},
          name="constraints")

## Step 4 — solve

In [ ]:
result = minimise_tracking_error(target, constraints, risk)

pd.Series({"converged": f"{result.diagnostics.converged!s}",
           "status": result.diagnostics.status,
           "iterations": f"{result.diagnostics.iterations}",
           "objective": f"{result.diagnostics.objective:.6f}"},
          name="solution")

### Which constraints actually bound?

This is the output that matters. `slack` is how much room was left — at the
bound, it is zero.

In [ ]:
if result.binding:
    display(pd.DataFrame([{"constraint": c.label, "kind": c.kind, "slack": c.slack}
                          for c in result.binding]).set_index("constraint"))
else:
    print("nothing bound: the unconstrained optimum already satisfied")
    print("everything asked of it, so the constraints cost nothing here")

These are the ones that shaped the answer. Relax any of them and the solution
moves; relax the others and nothing happens at all.

## Step 5 — what it did to the portfolio

In [ ]:
comparison = pd.DataFrame({"target": pd.Series(target),
                           "optimised": pd.Series(dict(result.weights))}).fillna(0.0)
comparison["active"] = comparison["optimised"] - comparison["target"]

comparison.sort_values("active", key=abs, ascending=False).head(10).style.format("{:.2%}")

In [ ]:
before = risk_contributions(target, risk.covariance)
after = risk_contributions(dict(result.weights), risk.covariance)

held = sum(1 for weight in dict(result.weights).values() if weight > 1e-6)

pd.Series({"target volatility": f"{before.volatility:.2%}",
           "optimised volatility": f"{after.volatility:.2%}",
           "names held": f"{held} of {len(target)}",
           "active share": f"{comparison['active'].abs().sum() / 2:.2%}"},
          name="effect")

The objective was **tracking error, not volatility**. A lower volatility here is
a side effect, not an achievement — optimising for one thing and reporting
another is the most common way an optimisation result gets misread.

### Where the risk actually sits

Risk contributions answer a different question from weights: not "how much do I
hold?" but "how much of the portfolio's volatility does this name explain?"

They sum to the portfolio volatility **exactly**, by Euler's theorem on
homogeneous functions — not approximately, and not by construction of the
reporting code.

In [ ]:
weights = pd.Series(dict(result.weights))

rows = pd.DataFrame({"weight": weights,
                     "marginal": pd.Series(after.marginal),
                     "contribution": pd.Series(after.contribution)}).dropna()
rows["share of risk"] = rows["contribution"] / after.volatility

rows = rows.sort_values("contribution", ascending=False)

print(f"contributions sum to  {rows['contribution'].sum():.8%}")
print(f"portfolio volatility  {after.volatility:.8%}   <-- identical, by Euler")
print(f"covered weight        {after.covered_weight:.2%}")

rows.head(8).style.format({"weight": "{:.2%}", "marginal": "{:.4f}",
                           "contribution": "{:.3%}", "share of risk": "{:.2%}"})

## Step 6 — proving the binding constraints matter

The claim above was that relaxing a binding constraint moves the solution and
relaxing a slack one does nothing. Here it is, tested rather than asserted.

In [ ]:
variants = {
    "as specified": constraints,
    "position 10%": [FullInvestment(), PositionBounds(0.0, 0.10),
                     GroupBounds(largest_sector, members, maximum=MAX_SECTOR)],
    "sector 40%": [FullInvestment(), PositionBounds(0.0, MAX_POSITION),
                   GroupBounds(largest_sector, members, maximum=0.40)],
    "full investment only": [FullInvestment()],
}

rows = []
for label, variant in variants.items():
    solved = minimise_tracking_error(target, variant, risk)
    weights = pd.Series(dict(solved.weights))
    active = weights.subtract(pd.Series(target), fill_value=0.0)

    rows.append({"constraints": label,
                 "objective": solved.diagnostics.objective,
                 "binding": len(solved.binding),
                 "active share": active.abs().sum() / 2,
                 "largest position": weights.max()})

pd.DataFrame(rows).set_index("constraints").style.format(
    {"objective": "{:.6f}", "active share": "{:.2%}",
     "largest position": "{:.2%}"})

Read the `objective` column. Every relaxation that lowers it was a constraint
that genuinely cost tracking error; any that leaves it unchanged was a
constraint the optimiser was never fighting.

The bottom row is the unconstrained-except-for-budget case — the best tracking
error achievable, and the benchmark the others should be judged against. The
distance from it is the price of the limits.

## Where to go next

- **`05_optimised_backtest.ipynb`** — backtest these weights and compare the
  risk model's forecast against what actually happened